In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Labs"
)

xpt_file = DATA_DIR / "FASTQX_E.xpt"
csv_file = DATA_DIR / "FASTQX_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 FASTQX_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "PHQ020",
    "PHACOFHR",
    "PHACOFMN",
    "PHQ030",
    "PHAALCHR",
    "PHAALCMN",
    "PHQ040",
    "PHAGUMHR",
    "PHAGUMMN",
    "PHQ050",
    "PHAANTHR",
    "PHAANTMN",
    "PHQ060",
    "PHASUPHR",
    "PHASUPMN",
    "PHAFSTHR",
    "PHAFSTMN",
    "PHDSESN"
]


if df.shape != (9307, 19):

    raise ValueError(
        f"Unexpected FASTQX_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "FASTQX_E columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():

    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: FASTQX_E structure = "
    "9,307 rows x 19 columns."
)


# ============================================================
# 4. KNOWN SAS/XPORT TINY ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)


print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)


print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print(
    "\nTiny values by variable:"
)


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


if int(
    true_zero_counts.sum()
) != 0:

    raise ValueError(
        "Unexpected true numeric zeros "
        "already present in source XPT."
    )


# ============================================================
# 5. EXPECTED TINY COUNTS
# ============================================================

EXPECTED_TINY_COUNTS = {

    "PHDSESN": 4311,

    "PHAFSTHR": 950,

    "PHAFSTMN": 161,

    "PHAGUMHR": 63,

    "PHACOFHR": 6,

    "PHAANTHR": 1,

    "PHASUPHR": 1,

    "PHASUPMN": 1
}


EXPECTED_TINY_TOTAL = 5494


# ============================================================
# 6. VALIDATE EXACT TINY PATTERN
# ============================================================

actual_tiny_counts = {
    col: int(count)
    for col, count in tiny_counts.items()
    if count > 0
}


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    missing_columns = (
        set(EXPECTED_TINY_COUNTS)
        -
        set(actual_tiny_counts)
    )


    unexpected_columns = (
        set(actual_tiny_counts)
        -
        set(EXPECTED_TINY_COUNTS)
    )


    count_differences = {
        col: {
            "expected": EXPECTED_TINY_COUNTS.get(col),
            "actual": actual_tiny_counts.get(col)
        }
        for col in (
            set(EXPECTED_TINY_COUNTS)
            &
            set(actual_tiny_counts)
        )
        if (
            EXPECTED_TINY_COUNTS[col]
            !=
            actual_tiny_counts[col]
        )
    }


    raise ValueError(
        "STOP: Tiny artifact pattern differs.\n"
        f"Missing variables: {sorted(missing_columns)}\n"
        f"Unexpected variables: {sorted(unexpected_columns)}\n"
        f"Different counts: {count_differences}"
    )


if int(
    tiny_counts.sum()
) != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny-value count."
    )


print(
    "\nPASS: Exactly 5,494 validated "
    "tiny XPORT artifacts found."
)


# ============================================================
# 7. SAFETY CHECK:
#    TINY VALUES ONLY IN CDC ZERO-PERMITTED VARIABLES
# ============================================================

APPROVED_ZERO_COLUMNS = {
    "PHACOFHR",
    "PHAGUMHR",
    "PHAANTHR",
    "PHASUPHR",
    "PHASUPMN",
    "PHAFSTHR",
    "PHAFSTMN",
    "PHDSESN"
}


unexpected_zero_columns = (
    set(actual_tiny_counts)
    -
    APPROVED_ZERO_COLUMNS
)


if unexpected_zero_columns:

    raise ValueError(
        "STOP: Tiny values found in "
        "unexpected FASTQX_E variables:\n"
        + str(
            sorted(
                unexpected_zero_columns
            )
        )
    )


print(
    "PASS: Tiny values occur only "
    "in validated zero-permitted fields."
)


# ============================================================
# 8. RESTORE ONLY VALIDATED ZEROS
# ============================================================

print(
    "\n--- ZERO RESTORATION ---"
)


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{actual_count:,} tiny -> 0"
    )


print(
    "PASS: All 5,494 validated zeros restored."
)


# ============================================================
# 9. VERIFY ZERO COUNTS AFTER RESTORATION
# ============================================================

print(
    "\n--- RESTORED ZERO COUNTS ---"
)


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<10}: "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: restored zero count differs."
        )


print(
    "PASS: Restored zero counts are exact."
)


# ============================================================
# 10. VERIFY NO TINY VALUE REMAINS
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifact remains."
    )


print(
    "PASS: No tiny artifacts remain."
)


# ============================================================
# 11. SEQN VALIDATION
# ============================================================

print(
    "\n--- SEQN CHECK ---"
)


seqn = df["SEQN"]


if int(
    seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn.min() != 41475:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "PASS: SEQN range = 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 12. GENUINE FRACTIONAL VALUE CHECK
# ============================================================

#
# IMPORTANT:
#
# Perform before converting to Int64.
#
# FASTQX_E contains NO genuine decimal variables.
#

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_mask = (
        ~np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    if fractional_count > 0:

        fractional_columns[
            col
        ] = fractional_count


print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)

print(
    "Genuine fractional variables:",
    fractional_columns
)


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal values found.\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: Genuine decimal variables = 0."
)


# ============================================================
# 13. CDC CHECK - PHQ020
# ============================================================

EXPECTED_PHQ020 = {
    1: 25,
    2: 8807
}


actual = (
    df["PHQ020"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_PHQ020:

    raise ValueError(
        "PHQ020 frequencies differ."
    )


if int(
    df["PHQ020"]
    .isna()
    .sum()
) != 475:

    raise ValueError(
        "PHQ020 missing count differs."
    )


print(
    "\nPASS: PHQ020 matches CDC."
)


# ============================================================
# 14. CDC CHECK - PHACOFHR
# ============================================================

#
# CDC:
#
# Range   = 0-12
# Valid   = 25
# Missing = 9,282
#
# Source zero count = 6
#

s = df["PHACOFHR"]


if not (
    int(
        s.notna()
        .sum()
    ) == 25

    and

    int(
        s.isna()
        .sum()
    ) == 9282

    and

    s.min() == 0

    and

    s.max() == 12

    and

    int(
        s.eq(0)
        .sum()
    ) == 6
):

    raise ValueError(
        "PHACOFHR differs from CDC/source."
    )


print(
    "PASS: PHACOFHR = 0-12, zeros=6."
)


# ============================================================
# 15. CDC CHECK - PHACOFMN
# ============================================================

s = df["PHACOFMN"]


if not (
    int(
        s.notna()
        .sum()
    ) == 25

    and

    int(
        s.isna()
        .sum()
    ) == 9282

    and

    s.min() == 2

    and

    s.max() == 58
):

    raise ValueError(
        "PHACOFMN differs from CDC."
    )


print(
    "PASS: PHACOFMN = 2-58."
)


# ============================================================
# 16. CDC CHECK - PHQ030
# ============================================================

EXPECTED_PHQ030 = {
    1: 7,
    2: 8825
}


actual = (
    df["PHQ030"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_PHQ030:

    raise ValueError(
        "PHQ030 frequencies differ."
    )


if int(
    df["PHQ030"]
    .isna()
    .sum()
) != 475:

    raise ValueError(
        "PHQ030 missing count differs."
    )


print(
    "PASS: PHQ030 matches CDC."
)


# ============================================================
# 17. CDC CHECK - ALCOHOL TIME
# ============================================================

s = df["PHAALCHR"]


if not (
    int(s.notna().sum()) == 7
    and
    int(s.isna().sum()) == 9300
    and
    s.min() == 1
    and
    s.max() == 15
):

    raise ValueError(
        "PHAALCHR differs from CDC."
    )


s = df["PHAALCMN"]


if not (
    int(s.notna().sum()) == 7
    and
    int(s.isna().sum()) == 9300
    and
    s.min() == 3
    and
    s.max() == 42
):

    raise ValueError(
        "PHAALCMN differs from CDC."
    )


print(
    "PASS: Alcohol fasting-time fields match CDC."
)


# ============================================================
# 18. CDC CHECK - PHQ040 / GUM TIMES
# ============================================================

EXPECTED_PHQ040 = {
    1: 88,
    2: 8744
}


actual = (
    df["PHQ040"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_PHQ040:

    raise ValueError(
        "PHQ040 frequencies differ."
    )


if int(
    df["PHQ040"]
    .isna()
    .sum()
) != 475:

    raise ValueError(
        "PHQ040 missing count differs."
    )


s = df["PHAGUMHR"]


if not (
    int(s.notna().sum()) == 88
    and
    int(s.isna().sum()) == 9219
    and
    s.min() == 0
    and
    s.max() == 15
    and
    int(s.eq(0).sum()) == 63
):

    raise ValueError(
        "PHAGUMHR differs from CDC/source."
    )


s = df["PHAGUMMN"]


if not (
    int(s.notna().sum()) == 88
    and
    int(s.isna().sum()) == 9219
    and
    s.min() == 1
    and
    s.max() == 57
):

    raise ValueError(
        "PHAGUMMN differs from CDC."
    )


print(
    "PASS: PHQ040 / gum fasting times match CDC."
)


# ============================================================
# 19. CDC CHECK - PHQ050 / ANTACID TIMES
# ============================================================

EXPECTED_PHQ050 = {
    1: 5,
    2: 8827
}


actual = (
    df["PHQ050"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_PHQ050:

    raise ValueError(
        "PHQ050 frequencies differ."
    )


if int(
    df["PHQ050"]
    .isna()
    .sum()
) != 475:

    raise ValueError(
        "PHQ050 missing count differs."
    )


s = df["PHAANTHR"]


if not (
    int(s.notna().sum()) == 5
    and
    int(s.isna().sum()) == 9302
    and
    s.min() == 0
    and
    s.max() == 8
    and
    int(s.eq(0).sum()) == 1
):

    raise ValueError(
        "PHAANTHR differs from CDC/source."
    )


s = df["PHAANTMN"]


if not (
    int(s.notna().sum()) == 5
    and
    int(s.isna().sum()) == 9302
    and
    s.min() == 9
    and
    s.max() == 30
):

    raise ValueError(
        "PHAANTMN differs from CDC."
    )


print(
    "PASS: PHQ050 / antacid times match CDC."
)


# ============================================================
# 20. CDC CHECK - PHQ060 / SUPPLEMENT TIMES
# ============================================================

EXPECTED_PHQ060 = {
    1: 16,
    2: 8816
}


actual = (
    df["PHQ060"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_PHQ060:

    raise ValueError(
        "PHQ060 frequencies differ."
    )


if int(
    df["PHQ060"]
    .isna()
    .sum()
) != 475:

    raise ValueError(
        "PHQ060 missing count differs."
    )


s = df["PHASUPHR"]


if not (
    int(s.notna().sum()) == 16
    and
    int(s.isna().sum()) == 9291
    and
    s.min() == 0
    and
    s.max() == 11
    and
    int(s.eq(0).sum()) == 1
):

    raise ValueError(
        "PHASUPHR differs from CDC/source."
    )


s = df["PHASUPMN"]


if not (
    int(s.notna().sum()) == 16
    and
    int(s.isna().sum()) == 9291
    and
    s.min() == 0
    and
    s.max() == 58
    and
    int(s.eq(0).sum()) == 1
):

    raise ValueError(
        "PHASUPMN differs from CDC/source."
    )


print(
    "PASS: PHQ060 / supplement times match CDC."
)


# ============================================================
# 21. CDC CHECK - TOTAL FASTING TIME
# ============================================================

s = df["PHAFSTHR"]


if not (
    int(s.notna().sum()) == 8832
    and
    int(s.isna().sum()) == 475
    and
    s.min() == 0
    and
    s.max() == 52
    and
    int(s.eq(0).sum()) == 950
):

    raise ValueError(
        "PHAFSTHR differs from CDC/source."
    )


s = df["PHAFSTMN"]


if not (
    int(s.notna().sum()) == 8832
    and
    int(s.isna().sum()) == 475
    and
    s.min() == 0
    and
    s.max() == 59
    and
    int(s.eq(0).sum()) == 161
):

    raise ValueError(
        "PHAFSTMN differs from CDC/source."
    )


print(
    "PASS: Total fasting hour/minute fields match CDC."
)


# ============================================================
# 22. CDC CHECK - PHDSESN
# ============================================================

#
# CDC:
#
# 0 Morning   = 4,311
# 1 Afternoon = 3,356
# 2 Evening   = 1,640
# Missing     = 0
#

EXPECTED_PHDSESN = {
    0: 4311,
    1: 3356,
    2: 1640
}


actual = (
    df["PHDSESN"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_PHDSESN:

    raise ValueError(
        "PHDSESN frequencies differ.\n"
        f"Expected: {EXPECTED_PHDSESN}\n"
        f"Actual:   {actual}"
    )


if int(
    df["PHDSESN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Unexpected PHDSESN missing values."
    )


print(
    "PASS: PHDSESN matches CDC."
)

print(
    "      Morning   = 4,311"
)

print(
    "      Afternoon = 3,356"
)

print(
    "      Evening   = 1,640"
)


# ============================================================
# 23. BUILD EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# expected_df =
#
# ORIGINAL XPT
#
# +
# ONLY these 5,494 validated tiny -> zero
# restorations.
#
# NOTHING ELSE.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 24. CLEAN WHOLE-NUMBER FORMATTING
# ============================================================

#
# We proved:
#
# Genuine decimal variables = 0
#
# Therefore all 19 released variables may safely
# use clean Int64 representation.
#
# Examples:
#
# SEQN:
# 41475.0 -> 41475
#
# PHAFSTHR:
# 12.0 -> 12
#
# PHDSESN:
# 0.0 -> 0
#
# Missing:
# NaN -> blank
#

for col in EXPECTED_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All FASTQX_E fields converted "
    "to clean integer formatting."
)


# ============================================================
# 25. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export numeric values equal "
    "expected XPT values exactly."
)


# ============================================================
# 26. EXPORT CSV
# ============================================================

#
# NO:
#
# float_format
# round()
# recalculation
#
# Missing SAS values -> blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 27. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 28. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure matches XPT."
)


# ============================================================
# 29. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 19 CSV columns numerically "
    "match expected XPT values exactly."
)


# ============================================================
# 30. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += differences


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 31. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(reader)


if len(raw_rows) != 9307:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 32. STRICT RAW CSV FORMAT FIDELITY
# ============================================================

#
# All FASTQX_E values are whole-number values.
#
# Expected raw representation:
#
# missing -> ""
#
# otherwise:
# str(int(expected_source_value))
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "FASTQX_E raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV formatting matches "
    "expected whole-number representation."
)


# ============================================================
# 33. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nValues ending '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        "Examples:",
        dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 34. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 35. FINAL TINY ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact remains in CSV."
)


# ============================================================
# 36. FINAL ZERO COUNT CHECK
# ============================================================

zero_failures = {}


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    if actual_count != expected_count:

        zero_failures[
            col
        ] = {
            "expected": expected_count,
            "actual": actual_count
        }


if zero_failures:

    raise ValueError(
        "Final zero counts changed:\n"
        + str(
            zero_failures
        )
    )


print(
    "PASS: All 5,494 restored zeros "
    "remain correct after export."
)


# ============================================================
# 37. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FASTQX_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nValidated tiny -> zero:"
)


for col, count in EXPECTED_TINY_COUNTS.items():

    print(
        f"{col:<10}: {count:,}"
    )


print(
    "TOTAL:",
    f"{EXPECTED_TINY_TOTAL:,}"
)


print(
    "\nUnexpected tiny variables:",
    0
)

print(
    "Genuine decimal variables:",
    0
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted .0 values:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "FASTQX_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 FASTQX_E XPT -> CSV
Rows: 9,307
Columns: 19
PASS: FASTQX_E structure = 9,307 rows x 19 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 5,494

Tiny values by variable:
PHDSESN     4311
PHAFSTHR     950
PHAFSTMN     161
PHAGUMHR      63
PHACOFHR       6
PHAANTHR       1
PHASUPHR       1
PHASUPMN       1

PASS: Exactly 5,494 validated tiny XPORT artifacts found.
PASS: Tiny values occur only in validated zero-permitted fields.

--- ZERO RESTORATION ---
PHDSESN   : 4,311 tiny -> 0
PHAFSTHR  : 950 tiny -> 0
PHAFSTMN  : 161 tiny -> 0
PHAGUMHR  : 63 tiny -> 0
PHACOFHR  : 6 tiny -> 0
PHAANTHR  : 1 tiny -> 0
PHASUPHR  : 1 tiny -> 0
PHASUPMN  : 1 tiny -> 0
PASS: All 5,494 validated zeros restored.

--- RESTORED ZERO COUNTS ---
PHDSESN   : 4,311
PHAFSTHR  : 950
PHAFSTMN  : 161
PHAGUMHR  : 63
PHACOFHR  : 6
PHAANTHR  : 1
PHASUPHR  : 1
PHASUPMN  : 1
PASS: Restored zero counts are exact.

Remaining tiny artifacts: 0
PASS: No tiny artifacts re